# 02 — POI Scraping Exploration

OSM via Overpass is the cheapest, most reproducible POI source for Sri Lanka. This notebook:

1. Tests one outlet end-to-end (query → response → features)
2. Sanity-checks the POI taxonomy choices against a real outlet's surroundings
3. Estimates run time so we can plan when to kick off the full ~20k × N-radii scrape
4. Validates the feature schema before letting the pipeline loose

> The actual full-fleet scrape is invoked from `run_pipeline.py` (or `python -m src.features.poi_scraper`).

In [ ]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.features.poi_scraper import (
    build_overpass_query, fetch_outlet_pois, build_poi_features,
    _features_from_response,
)
from src.utils.io import read_parquet

## 1. Inspect the Overpass query template

We hand-verify one query in the Overpass turbo web UI before trusting the batch run.

In [ ]:
colombo_lat, colombo_lon = 6.9271, 79.8612
sample_query = build_overpass_query(
    colombo_lat, colombo_lon, radius_m=1000,
    taxonomy=config.POI_CONFIG["poi_taxonomy"],
)
print(sample_query)

## 2. Live test on one outlet

Run a single Overpass call live; verify the JSON shape and that our classifier picks up the expected categories.

In [ ]:
t0 = time.time()
resp = fetch_outlet_pois(
    "TEST_COLOMBO_FORT", colombo_lat, colombo_lon, 1000,
    config.POI_CONFIG["poi_taxonomy"],
)
elapsed = time.time() - t0
print(f"elapsed: {elapsed:.1f}s | elements: {len(resp['elements']) if resp else 'NONE'}")

feats = _features_from_response(
    "TEST_COLOMBO_FORT", colombo_lat, colombo_lon, 1000, resp,
    config.POI_CONFIG["poi_taxonomy"],
)
import pprint; pprint.pprint(feats)

## 3. Pilot scrape (10 outlets)

Before launching the full scrape we run a small pilot to estimate throughput, exercise the retry path, and confirm caching works.

In [ ]:
silver_outlets = config.SILVER_DIR / "outlets.parquet"
if silver_outlets.exists():
    outlets_df = read_parquet(silver_outlets)
    pilot = build_poi_features(outlets_df=outlets_df, limit=10)
    display(pilot.head())
else:
    print(f"Run silver_cleaning first; expected {silver_outlets}")